In [7]:
%pip install -q "stable-baselines3[extra]"


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env

import kuka_dynamics

# ------------------------------------------------------------
# 1. JOINT LIMITS AND PICK-PLACE SETTINGS
# ------------------------------------------------------------
LOW = kuka_dynamics.JOINT_LOW
HIGH = kuka_dynamics.JOINT_HIGH
GRIP_DISTANCE = 10.0
SUCCESS_DISTANCE = 24.0

# ------------------------------------------------------------
# 2. DISCRETE PICK-AND-PLACE ENVIRONMENT FOR DDQN
# ------------------------------------------------------------
class DiscreteJointReachEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, max_steps=360, dt=0.05, max_velocity=0.65):
        super().__init__()
        self.max_steps = max_steps
        self.dt = dt
        self.max_velocity = max_velocity
        self.action_space = spaces.Discrete(12)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(18,), dtype=np.float32)

    def _obs(self):
        return kuka_dynamics.make_observation(self.q, self.goal, self.prev_v)

    def _sample_reachable_position(self):
        for _ in range(5000):
            position = self.np_random.uniform(
                low=np.array([-180.0, -180.0, 10.0]),
                high=np.array([180.0, 180.0, 80.0]),
            ).astype(np.float32)
            if kuka_dynamics.solve_ik(position, self.q) is not None:
                return position
        raise RuntimeError("Could not sample a reachable object/tray position")

    def _joint_goal(self, position):
        goal = kuka_dynamics.solve_ik(position, self.q)
        if goal is None:
            raise ValueError(f"Position is outside the arm workspace: {position}")
        return np.asarray(goal, dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.steps = 0
        self.q = self.np_random.uniform(LOW * 0.35, HIGH * 0.35).astype(np.float32)
        options = options or {}

        object_position = options.get("object_position")
        tray_position = options.get("tray_position")
        self.object_position = (
            np.asarray(object_position, dtype=np.float32)
            if object_position is not None
            else self._sample_reachable_position()
        )
        self.tray_position = (
            np.asarray(tray_position, dtype=np.float32)
            if tray_position is not None
            else self._sample_reachable_position()
        )
        for name, position in (("object_position", self.object_position), ("tray_position", self.tray_position)):
            if position.shape != (3,) or not np.isfinite(position).all():
                raise ValueError(f"{name} must contain three finite XYZ coordinates")
            if kuka_dynamics.solve_ik(position, self.q) is None:
                raise ValueError(f"{name} is outside the arm workspace: {position}")
        if object_position is None and tray_position is None:
            while np.linalg.norm(self.object_position - self.tray_position) <= SUCCESS_DISTANCE:
                self.tray_position = self._sample_reachable_position()

        self.prev_v = np.zeros(6, dtype=np.float32)
        self.is_holding = False
        self.has_picked = False
        self.goal = self._joint_goal(self.object_position)
        return self._obs(), {"object_position": self.object_position.copy(), "tray_position": self.tray_position.copy()}

    def step(self, action):
        self.steps += 1
        velocity = np.zeros(6, dtype=np.float32)
        joint_idx = int(action) // 2
        direction = 1.0 if int(action) % 2 == 0 else -1.0
        velocity[joint_idx] = direction * self.max_velocity

        old_dist = float(np.linalg.norm(self.goal - self.q))
        self.q = np.clip(self.q + velocity * self.dt, LOW, HIGH).astype(np.float32)
        new_dist = float(np.linalg.norm(self.goal - self.q))
        end_effector = np.asarray(kuka_dynamics.forward_kinematics(self.q)[-1])
        target = self.tray_position if self.is_holding else self.object_position
        target_distance = float(np.linalg.norm(end_effector - target))

        smooth_penalty = float(np.mean((velocity - self.prev_v) ** 2))
        speed_penalty = float(np.mean(velocity ** 2))
        reward = 12 * (old_dist - new_dist) - 0.035 * smooth_penalty - 0.004 * speed_penalty - 0.015
        success = False
        if self.is_holding:
            self.object_position = end_effector.copy()
            if target_distance <= SUCCESS_DISTANCE:
                self.object_position = self.tray_position.copy()
                self.is_holding = False
                success = True
                reward += 20.0
        elif target_distance <= GRIP_DISTANCE:
            self.object_position = end_effector.copy()
            self.is_holding = True
            self.has_picked = True
            self.goal = self._joint_goal(self.tray_position)
            reward += 8.0

        self.prev_v = velocity.copy()
        terminated = success
        truncated = self.steps >= self.max_steps and not terminated
        info = {
            "distance": new_dist,
            "target_distance": target_distance,
            "reached": success,
            "phase": "place" if self.is_holding else ("complete" if success else "pick"),
            "object_position": self.object_position.copy(),
            "tray_position": self.tray_position.copy(),
        }
        return self._obs(), float(reward), terminated, truncated, info

# ------------------------------------------------------------
# 3. DDQN TRAINING IMPLEMENTATION
# ------------------------------------------------------------
if __name__ == "__main__":
    env = DiscreteJointReachEnv()
    check_env(env, warn=True)
    print("Pick-and-place environment ready:", env.observation_space.shape, env.action_space.n)

    model = DQN(
        "MlpPolicy",
        env,
        learning_rate=1e-3,
        buffer_size=100000,
        learning_starts=1000,
        batch_size=128,
        tau=0.01,
        gamma=0.98,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=1000,
        exploration_fraction=0.3,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.05,
        policy_kwargs=dict(net_arch=[256, 256]),
        verbose=1,
        device="auto"
    )

    print("Training DDQN for movable object/tray positions...")
    model.learn(total_timesteps=200_000, progress_bar=True)
    model.save("ddqn_6dof_arm_pick_place")
    print("Model saved to ddqn_6dof_arm_pick_place.zip")

    # ------------------------------------------------------------
    # 4. EVALUATION ROLLOUT
    # ------------------------------------------------------------
    obs, info = env.reset()
    for step in range(env.max_steps):
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            print(
                f"Evaluation complete. Success: {info['reached']}; "
                f"object={info['object_position']}; tray={info['tray_position']}"
            )
            break

Pick-and-place environment ready: (18,) 12
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/opt/homebrew/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Training DDQN for movable object/tray positions...
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 360      |
|    ep_rew_mean      | -5.67    |
|    exploration_rate | 0.977    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 3940     |
|    time_elapsed     | 0        |
|    total_timesteps  | 1440     |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.000112 |
|    n_updates        | 109      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 360      |
|    ep_rew_mean      | -5.25    |
|    exploration_rate | 0.954    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 3816     |
|    time_elapsed     | 0        |
|    total_timesteps  | 2880     |
| train/              |          |
|    learning_rate    | 0.001    |
|   

Model saved to ddqn_6dof_arm_pick_place.zip
Evaluation complete. Success: True; object=[165.08838  -41.352127  58.386993]; tray=[165.08838  -41.352127  58.386993]


In [9]:
from stable_baselines3.common.evaluation import evaluate_policy

# Evaluate the trained model for 100 episodes
mean_reward, std_reward = evaluate_policy(
    model,
    env,
    n_eval_episodes=100,
    deterministic=True # True means it uses the best actions without random exploration
)


import numpy as np

eval_episodes = 100
success_count = 0
final_distances = []

print(f"Starting evaluation over {eval_episodes} episodes...")

for episode in range(eval_episodes):
    obs, info = env.reset()
    done = False

    while not done:
        # Predict the best action
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

    # Record the results at the end of the episode
    if info.get("reached", False):
        success_count += 1

    final_distances.append(info.get("distance", 0.0))

# Calculate final metrics
success_rate = (success_count / eval_episodes) * 100
mean_distance = np.mean(final_distances)
min_distance = np.min(final_distances)

print("\n" + "="*40)
print("EVALUATION RESULTS")
print("="*40)
print(f"Success Rate:           {success_rate:.1f}% ({success_count}/{eval_episodes})")
print(f"Average Final Distance: {mean_distance:.4f} m")
print(f"Best Final Distance:    {min_distance:.4f} m")
print("="*40)


/opt/homebrew/lib/python3.11/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Starting evaluation over 100 episodes...

EVALUATION RESULTS
Success Rate:           92.0% (92/100)
Average Final Distance: 0.3357 m
Best Final Distance:    0.1736 m
